# シンドロームの計算とサンプル

<a href="https://github.com/Tsukumo-999/qr-code-from-scratch/blob/master/Step3/1_syndrome.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

第8回記事「[QRコードを読み解く：シンドロームの計算とエラー検出【QRコードを解読する 第8回】](https://tukumolog.com/qr-code-decode-08-syndrome-calculation/)」で行った、誤り訂正に使うシンドロームの計算です

### 1. サンプルのデータのエンコード

HALという文字列を入れていたとします。以下で、エンコードします

In [14]:
num_ec_codewords = 4
word = "HAL"

# ==========================================
# 1. ガロア体 GF(2^8) の準備
# ==========================================
def generate_galois_tables():
    exp = [0] * 256
    log = [0] * 256
    v = 1
    for i in range(255):
        exp[i] = v
        log[v] = i
        v <<= 1
        if v & 0x100:
            v ^= 0x11D
    exp[255] = exp[0]
    return exp, log

exp_table, log_table = generate_galois_tables()

def gf_mul(x, y):
    """ガロア体 GF(2^8) 上での乗算"""
    if x == 0 or y == 0:
        return 0
    return exp_table[(log_table[x] + log_table[y]) % 255]

# ==========================================
# 2. 生成多項式 G(x) の計算
# ==========================================
G = [1]
for i in range(num_ec_codewords):
    alpha_i = exp_table[i]
    new_G = [0] * (len(G) + 1)
    for j, coef in enumerate(G):
        new_G[j] ^= coef # x を掛ける
        new_G[j + 1] ^= gf_mul(coef, alpha_i) # α^i を掛ける
    G = new_G

# ==========================================
# 3. データコード語の作成
# ==========================================
# 文字列をASCIIコード（整数）の配列に直接変換します
data_codewords = [ord(char) for char in word]

print(f"データコード語: {data_codewords}")

# ==========================================
# 4. 誤り訂正コードの計算 (多項式の割り算)
# ==========================================
# 後ろに誤り訂正コードを入れるための器（4個の0）を用意
msg = data_codewords + [0] * num_ec_codewords

for i in range(len(data_codewords)):
    coef = msg[i]
    if coef != 0:
        # G(x) の各項を coef 倍して、msg から引く（XOR）
        for j in range(len(G)):
            msg[i + j] ^= gf_mul(G[j], coef)

# msgの後ろ4バイトに「余り」が残るので、それを抽出
ec_codewords = msg[-num_ec_codewords:]

print(f"誤り訂正コード: {ec_codewords}")

# ==========================================
# 5. 完成したメッセージの結合
# ==========================================
final_message = data_codewords + ec_codewords
print("\n▼ 送信メッセージ（完璧に正しい配列） ▼")
print(final_message)

データコード語: [72, 65, 76]
誤り訂正コード: [97, 162, 112, 246]

▼ 送信メッセージ（完璧に正しい配列） ▼
[72, 65, 76, 97, 162, 112, 246]


## 2. 誤りを起こす

デコードの実験をするために、このメッセージにわざとエラーを起こします。
2番目の文字「A（65）」の部分に汚れが付着し、「99」という間違った数値に化けてしまったと仮定しましょう。

In [17]:
# 正しいメッセージ
correct_msg = [72, 65, 76, 97, 162, 112, 246]

# 3つ目を99に置き換える
received_message = correct_msg
received_message[2] = 99

# 破損したデータ
print("誤ったデータ",received_message)


誤ったデータ [72, 65, 99, 97, 162, 112, 246]


## 3. Pythonによるシンドローム計算の実装
上のエンコードで作成した完璧な配列に対して、2番目の文字「A（65）」を「99」に書き換えた**ノイズ入り受信データ**を作成できたので。

この受信データ $R(x)$ に対して、ホーナー法を用いて $x$ に $\alpha^0, \alpha^1, \alpha^2, \alpha^3$ を代入し、4つのシンドローム $S_0, S_1, S_2, S_3$ を計算してみましょう。

In [18]:
# --- シンドロームの計算 ---
num_ec_codewords = 4
syndromes = []

for i in range(num_ec_codewords):
    alpha_i = exp_table[i]  # 代入する値 (α^0, α^1, α^2, α^3)
    
    syndrome = 0
    # ホーナー法を用いた多項式の代入計算
    for coef in received_message:
        syndrome = coef ^ gf_mul(syndrome, alpha_i)
        
    syndromes.append(syndrome)

print("▼ 計算されたシンドローム ▼")
for i, s in enumerate(syndromes):
    print(f" S_{i} = {s}")

▼ 計算されたシンドローム ▼
 S_0 = 47
 S_1 = 202
 S_2 = 60
 S_3 = 231


計算結果がすべて「0」ではないため、確実にデータが壊れていることが検知できました。<br>
このシンドロームは、単なるエラーの「兆候」ではありません。エラーが「1箇所だけ」の場合、シンドロームの中に犯人の正体がそのまま隠されています。

## 4. エラーが1個なら「割り算」で場所と値がわかる！
エラーの値を $e$、エラーの起きた場所（後ろからの位置）を $\alpha^j$ とすると、最初の2つのシンドロームは以下の式になります。

* $S_0 = e \cdot (\alpha^j)^0 = e$
* $S_1 = e \cdot (\alpha^j)^1 = e \cdot \alpha^j$

つまり、$S_0$ がそのまま **「エラーの値（反転の強さ）」** を示しています。
さらに、$S_1$ を $S_0$ でガロア体の割り算を行うと、値 $e$ が打ち消され、**「エラーの場所 $\alpha^j$」** だけが綺麗に浮かび上がります！

さっそく、ガロア体用の割り算関数 `gf_div` を追加して、犯人を特定・修復してみましょう。

In [20]:
# ガロア体 GF(2^8) 上での割り算関数を定義
def gf_div(x, y):
    if y == 0:
        raise ZeroDivisionError("0で割ることはできません")
    if x == 0:
        return 0
    return exp_table[(log_table[x] - log_table[y]) % 255]

# --- エラーの特定と修復 ---
S0 = syndromes[0]
S1 = syndromes[1]

# 1. エラーの値は S0 そのもの
error_value = S0

# 2. S1 / S0 を計算してエラーの位置 (α^j) を特定
error_pos_alpha = gf_div(S1, S0)

# αの何乗か（j）を対数表から取得
j = log_table[error_pos_alpha]

# 多項式は x^6, x^5... と並んでいるため、配列のインデックスに変換
error_index = (len(received_message) - 1) - j

print(f"【解析結果】")
print(f"・エラーの値 (e): {error_value}")
print(f"・エラーの場所 (j): 後ろから {j} 乗の位置")
print(f"・配列のインデックス: {error_index} 番目\n")

# --- データの修復 ---
repaired_message = received_message.copy()
# 判明した場所に対して、判明したエラー値をXOR（排他的論理和）して反転を元に戻す
repaired_message[error_index] ^= error_value

print("▼ 修復結果 ▼")
print(f"壊れたメッセージ: {received_message}")
print(f"修復後メッセージ: {repaired_message}")

# 文字列に戻して確認
decoded_chars = [chr(c) for c in repaired_message[:3]]
print(f"\n復元された文字列: {''.join(decoded_chars)}")
if repaired_message == correct_msg:
    print("🎉 完璧に修復されました！")

【解析結果】
・エラーの値 (e): 47
・エラーの場所 (j): 後ろから 4 乗の位置
・配列のインデックス: 2 番目

▼ 修復結果 ▼
壊れたメッセージ: [72, 65, 99, 97, 162, 112, 246]
修復後メッセージ: [72, 65, 76, 97, 162, 112, 246]

復元された文字列: HAL


##  5. ２か所破損していると復元できない？

たった1つのエラーであれば、「シンドロームの割り算」という非常にシンプルな計算だけで、見事にエラーの場所と値を特定し、元の「HAL」を復元することができました！

数学の計算の力だけで、傷ついたデータがパズルを解くように元通りになる快感を味わえたのではないでしょうか。

### しかし、エラーが2つになると…？
これはあくまで「エラーが1つだけ」という限定的な状況（ラッキーパターン）です。
もし、エラーが2箇所（値 $e_1, e_2$、場所 $\alpha^i, \alpha^j$）に増えると、シンドロームの式はどうなるでしょうか？

* $S_0 = e_1 + e_2$
* $S_1 = e_1\alpha^i + e_2\alpha^j$

お分かりでしょうか。このように変数が混ざり合ってしまうため、$S_1 \div S_0$ のように**単純な割り算をしても、綺麗に打ち消し合ってくれません。**
未知数が4つ（2つの場所と、2つの値）もある方程式を、どうやって解けばいいのでしょうか？

そこで次回は、複数のエラーが複雑に絡み合ったシンドロームから、「エラーの場所を示す方程式（エラー位置多項式）」をシステマチックに解き明かす最強のアルゴリズム、「バーレカンプ・マッシー（BM）法」に挑みます！

過去の予測の失敗を学習しながら、正しい方程式をジワジワと「育てていく」という、プログラミング的にも非常に胸熱なアルゴリズムです。

[バーレカンプ・マッシー（BM）法の解説](https://tukumolog.com/qr-code-decode-berlekamp-massey/)

## 6.参考資料とリンク 

本記事のコード実装およびQRコードの仕様解読にあたり、以下の文献やサイトを参考にしています。より深く原理を知りたい方はぜひご参照ください。

### 当シリーズの過去記事
* **[QRコードの作り方を徹底解説｜"HELLO"を例にエンコーダをゼロから実装する](https://tukumolog.com/qr-code-encode-tutorial/)**
  * 本記事の前編となるエンコード処理の解説記事です。データの埋め込みやリードソロモン符号の付加過程をステップバイステップで解説しています。
* **[デジタル世界の数学「ガロア体」を超ざっくり理解しよう！](https://tukumolog.com/reed-solomon-galois-field-basics/)**
  * 今回のデコード計算でも活躍した「ガロア体（GF(2^8)）」の基礎理論を分かりやすく解説しています。
* **[QRコードをデコードする（Happy Path）【QRコードを解読する 第7回】](https://tukumolog.com/qr-code-decode-07-hello-recovery/)**
  * 誤りがない場合のデコードを試してみます。

### 公式規格
* **[JIS X 0510:2018 (ISO/IEC 18004:2015) 二次元コードシンボル－QRコード－基本仕様](https://kikakurui.com/x0/X0510-2018-01.html)**
  * QRコードの公式な規格書（JIS規格）です。マスクパターンの計算式やデータ配置の厳密なルールについてはこちらを参照しています。

### 専門書・技術解説（リードソロモン符号・ガロア体）
* **[ガロア体入門 (Theoretical Background)](https://theoretical-background.com/%E3%82%AC%E3%83%AD%E3%82%A2%E4%BD%93%E5%85%A5%E9%96%80/)**
  * ガロア体上の多項式演算など、数学的背景について非常に詳しくまとめられているサイトです。

### 次回以降のアルゴリズム解説
* **[バーレカンプ・マッシー（BM）法の解説](https://tukumolog.com/qr-code-decode-berlekamp-massey/)**
  * 本記事の最後で触れた2つ以上の欠損があるとき、欠損位置をみつけるアルゴリズムです。